# Base de datos del catálogo de Netflix con SQL

Este notebook crea una base de datos relacional con SQLite (incluida en Python, no necesita instalación), carga el catálogo de Netflix y responde preguntas con consultas SQL.

**Qué demuestra:** diseño de tablas con claves primarias y foráneas, normalización de una columna con varios valores (países), carga de datos y consultas con `SELECT`, `JOIN`, `GROUP BY`, `HAVING`, `CASE` y funciones de fecha.

**Requisito:** el archivo `netflix_titles.csv` debe estar en la carpeta `/content` de Colab.

## 1. Conexión y configuración

In [5]:
import os
import sqlite3
import pandas as pd

RUTA_CSV = "netflix_titles.csv"
RUTA_BD = "netflix.db"

# Si la base ya existe, la borramos para empezar limpio
if os.path.exists(RUTA_BD):
    os.remove(RUTA_BD)

conexion = sqlite3.connect(RUTA_BD)
conexion.execute("PRAGMA foreign_keys = ON")  # activa las claves foráneas
cursor = conexion.cursor()
print("Conectado a", RUTA_BD)

Conectado a netflix.db


## 2. Diseño de las tablas

Usamos tres tablas:
- **titulos:** una fila por película o serie. La clave primaria es `show_id`.
- **paises:** una fila por país, sin repetir nombres.
- **titulo_pais:** tabla intermedia que relaciona cada título con sus países. Como un título puede tener varios países, esta relación es de muchos a muchos.

In [6]:
cursor.executescript("""
CREATE TABLE titulos (
    show_id        TEXT PRIMARY KEY,
    tipo           TEXT NOT NULL CHECK (tipo IN ('Movie', 'TV Show')),
    titulo         TEXT NOT NULL,
    director       TEXT,
    anio_estreno   INTEGER,
    clasificacion  TEXT,
    duracion       TEXT,
    fecha_agregado TEXT,
    descripcion    TEXT
);

CREATE TABLE paises (
    id_pais  INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre   TEXT NOT NULL UNIQUE
);

CREATE TABLE titulo_pais (
    show_id  TEXT    NOT NULL,
    id_pais  INTEGER NOT NULL,
    PRIMARY KEY (show_id, id_pais),
    FOREIGN KEY (show_id) REFERENCES titulos(show_id),
    FOREIGN KEY (id_pais) REFERENCES paises(id_pais)
);
""")
print("Tablas creadas:")
print(cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall())

Tablas creadas:
[('titulos',), ('paises',), ('sqlite_sequence',), ('titulo_pais',)]


## 3. Lectura y limpieza del CSV con pandas

In [7]:
df = pd.read_csv(RUTA_CSV)
df["country"] = df["country"].fillna("Desconocido")
df["director"] = df["director"].fillna("Sin dato")
df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), errors="coerce")

print("Filas leídas:", len(df))

Filas leídas: 8807


## 4. Carga de la tabla titulos

In [8]:
filas_titulos = [
    (
        r.show_id,
        r.type,
        r.title,
        r.director,
        int(r.release_year),
        r.rating if pd.notna(r.rating) else None,
        r.duration if pd.notna(r.duration) else None,
        r.date_added.strftime("%Y-%m-%d") if pd.notna(r.date_added) else None,
        r.description,
    )
    for r in df.itertuples(index=False)
]

cursor.executemany("""
    INSERT INTO titulos (show_id, tipo, titulo, director, anio_estreno,
                         clasificacion, duracion, fecha_agregado, descripcion)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""", filas_titulos)
conexion.commit()

print("Títulos insertados:", cursor.execute("SELECT COUNT(*) FROM titulos").fetchone()[0])

Títulos insertados: 8807


## 5. Carga de países y relación título-país

Separamos los países que vienen juntos en el CSV ("United States, Canada") y guardamos cada uno en su propia fila.

In [9]:
paises_unicos = set()
pares = []
for show_id, texto in zip(df["show_id"], df["country"]):
    for pais in texto.split(","):
        pais = pais.strip()
        paises_unicos.add(pais)
        pares.append((show_id, pais))

cursor.executemany(
    "INSERT INTO paises (nombre) VALUES (?)",
    [(p,) for p in sorted(paises_unicos)],
)

# Diccionario nombre -> id, para llenar la tabla intermedia
mapa = dict(cursor.execute("SELECT nombre, id_pais FROM paises").fetchall())

cursor.executemany(
    "INSERT OR IGNORE INTO titulo_pais (show_id, id_pais) VALUES (?, ?)",
    [(sid, mapa[p]) for sid, p in pares],
)
conexion.commit()

print("Países:", cursor.execute("SELECT COUNT(*) FROM paises").fetchone()[0])
print("Relaciones título-país:", cursor.execute("SELECT COUNT(*) FROM titulo_pais").fetchone()[0])

Países: 124
Relaciones título-país: 10850


## 6. Verificación de integridad
Si la lista sale vacía, no hay claves foráneas rotas.

In [10]:
print("Errores de claves foráneas:", cursor.execute("PRAGMA foreign_key_check").fetchall())

Errores de claves foráneas: []


## 7. Consultas SQL
Cada consulta responde una pregunta sobre el catálogo.

In [11]:
def consulta(sql):
    """Ejecuta una consulta SELECT y devuelve el resultado como tabla."""
    return pd.read_sql_query(sql, conexion)

### Consulta 1: cantidad de títulos por tipo

In [12]:
consulta("""
SELECT tipo, COUNT(*) AS total
FROM titulos
GROUP BY tipo
ORDER BY total DESC;
""")

,tipo,total
0,Movie,6131
1,TV Show,2676


### Consulta 2: los 10 países con más títulos
Usamos `JOIN` para unir la tabla intermedia con la de países, y excluimos los países desconocidos.

In [13]:
consulta("""
SELECT p.nombre AS pais, COUNT(*) AS titulos
FROM titulo_pais tp
JOIN paises p ON p.id_pais = tp.id_pais
WHERE p.nombre <> 'Desconocido'
GROUP BY p.nombre
ORDER BY titulos DESC
LIMIT 10;
""")

,pais,titulos
0,United States,3690
1,India,1046
2,United Kingdom,806
3,Canada,445
4,France,393
5,Japan,318
6,Spain,232
7,South Korea,231
8,Germany,226
9,Mexico,169


### Consulta 3: estrenos por año desde 2000

In [14]:
consulta("""
SELECT anio_estreno, COUNT(*) AS titulos
FROM titulos
WHERE anio_estreno >= 2000
GROUP BY anio_estreno
ORDER BY anio_estreno;
""")

,anio_estreno,titulos
0,2000,37
1,2001,45
2,2002,51
3,2003,61
4,2004,64
5,2005,80
6,2006,96
7,2007,88
8,2008,136
9,2009,152


### Consulta 4: años con más de 500 estrenos
`HAVING` filtra los grupos después de agruparlos, a diferencia de `WHERE`, que filtra filas antes.

In [15]:
consulta("""
SELECT anio_estreno, COUNT(*) AS titulos
FROM titulos
GROUP BY anio_estreno
HAVING COUNT(*) > 500
ORDER BY titulos DESC;
""")

,anio_estreno,titulos
0,2018,1147
1,2017,1032
2,2019,1030
3,2020,953
4,2016,902
5,2021,592
6,2015,560


### Consulta 5: películas de Estados Unidos estrenadas desde 2015

In [16]:
consulta("""
SELECT t.titulo, t.anio_estreno
FROM titulos t
JOIN titulo_pais tp ON tp.show_id = t.show_id
JOIN paises p ON p.id_pais = tp.id_pais
WHERE p.nombre = 'United States'
  AND t.tipo = 'Movie'
  AND t.anio_estreno >= 2015
ORDER BY t.anio_estreno DESC, t.titulo
LIMIT 15;
""")

,titulo,anio_estreno
0,7 Yards: The Chris Norton Story,2021
1,A Week Away,2021
2,Accomplice,2021
3,Aftermath,2021
4,America: The Motion Picture,2021
5,Amy Tan: Unintended Memoir,2021
6,Animals on the Loose: A You vs. Wild Movie,2021
7,Arlo the Alligator Boy,2021
8,Army of the Dead,2021
9,Awake,2021


### Consulta 6: clasificaciones más frecuentes

In [17]:
consulta("""
SELECT clasificacion, COUNT(*) AS titulos
FROM titulos
WHERE clasificacion IS NOT NULL
GROUP BY clasificacion
ORDER BY titulos DESC
LIMIT 5;
""")

,clasificacion,titulos
0,TV-MA,3207
1,TV-14,2160
2,TV-PG,863
3,R,799
4,PG-13,490


### Consulta 7: directores con más títulos

In [18]:
consulta("""
SELECT director, COUNT(*) AS titulos
FROM titulos
WHERE director <> 'Sin dato'
GROUP BY director
ORDER BY titulos DESC
LIMIT 10;
""")

,director,titulos
0,Rajiv Chilaka,19
1,"Raúl Campos, Jan Suter",18
2,Suhas Kadav,16
3,Marcus Raboy,16
4,Jay Karas,14
5,Cathy Garcia-Molina,13
6,Youssef Chahine,12
7,Martin Scorsese,12
8,Jay Chapman,12
9,Steven Spielberg,11


### Consulta 8: países con más de 100 títulos y porcentaje de series
`CASE` permite contar series dentro de la misma consulta, sin hacer dos consultas separadas.

In [19]:
consulta("""
SELECT p.nombre AS pais,
       COUNT(*) AS total,
       SUM(CASE WHEN t.tipo = 'TV Show' THEN 1 ELSE 0 END) AS series,
       ROUND(100.0 * SUM(CASE WHEN t.tipo = 'TV Show' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_series
FROM titulo_pais tp
JOIN titulos t ON t.show_id = tp.show_id
JOIN paises p ON p.id_pais = tp.id_pais
WHERE p.nombre <> 'Desconocido'
GROUP BY p.nombre
HAVING COUNT(*) > 100
ORDER BY total DESC;
""")

,pais,total,series,pct_series
0,United States,3690,938,25.4
1,India,1046,84,8.0
2,United Kingdom,806,272,33.7
3,Canada,445,126,28.3
4,France,393,90,22.9
5,Japan,318,199,62.6
6,Spain,232,61,26.3
7,South Korea,231,170,73.6
8,Germany,226,44,19.5
9,Mexico,169,58,34.3


### Consulta 9: títulos agregados al catálogo por mes (2019-2020)
`strftime` es una función de fecha de SQLite. En PostgreSQL se usaría `TO_CHAR`.

In [20]:
consulta("""
SELECT strftime('%Y-%m', fecha_agregado) AS mes, COUNT(*) AS titulos
FROM titulos
WHERE fecha_agregado BETWEEN '2019-01-01' AND '2020-12-31'
GROUP BY mes
ORDER BY mes;
""")

,mes,titulos
0,2019-01,153
1,2019-02,148
2,2019-03,172
3,2019-04,162
4,2019-05,139
5,2019-06,168
6,2019-07,157
7,2019-08,131
8,2019-09,123
9,2019-10,193


### Consulta 10: títulos sin país registrado

In [21]:
consulta("""
SELECT COUNT(DISTINCT tp.show_id) AS titulos_sin_pais
FROM titulo_pais tp
JOIN paises p ON p.id_pais = tp.id_pais
WHERE p.nombre = 'Desconocido';
""")

,titulos_sin_pais
0,831


## 8. Cierre

In [22]:
conexion.close()
print("Conexión cerrada. La base quedó guardada en", RUTA_BD)

Conexión cerrada. La base quedó guardada en netflix.db
